In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

project_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_path / "data"
model_path = project_path / "models"

data_path.mkdir(parents=True, exist_ok=True)
model_path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path / "dataset.csv")

print("Original dataset shape:", df.shape)

if "label" not in df.columns:
    raise ValueError("Dataset must contain a 'label' column.")

if df["label"].isnull().any():
    raise ValueError("Target labels contain missing values. Check the dataset before training.")

if "patient_id" in df.columns:
    df = df.drop(columns=["patient_id"])

X = df.drop(columns=["label"])
y = df["label"]

if not all(pd.api.types.is_numeric_dtype(X[col]) for col in X.columns):
    raise ValueError("All feature columns must be numeric. Encode categorical features first.")

if y.nunique() < 2:
    raise ValueError("The target must contain at least two classes.")

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
display(y.value_counts().sort_index().to_frame("Count"))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

feature_columns = X.columns.tolist()

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns,
    index=X_test.index
)

print("\nTRAIN/TEST SPLIT")
print("Training features:", X_train_scaled.shape)
print("Testing features:", X_test_scaled.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("\nTraining label percentages:")
display((y_train.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("Percentage"))

print("\nTesting label percentages:")
display((y_test.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("Percentage"))

X_train_scaled.to_csv(data_path / "X_train.csv", index=False)
X_test_scaled.to_csv(data_path / "X_test.csv", index=False)
y_train.to_frame("label").to_csv(data_path / "y_train.csv", index=False)
y_test.to_frame("label").to_csv(data_path / "y_test.csv", index=False)

joblib.dump(imputer, model_path / "imputer.joblib")
joblib.dump(scaler, model_path / "scaler.joblib")
joblib.dump(feature_columns, model_path / "feature_columns.joblib")

print("\nSaved files:")
for filename in [
    "X_train.csv",
    "X_test.csv",
    "y_train.csv",
    "y_test.csv"
]:
    print(data_path / filename)

for filename in [
    "imputer.joblib",
    "scaler.joblib",
    "feature_columns.joblib"
]:
    print(model_path / filename)

print("\nData preprocessing completed.")

Original dataset shape: (5000, 8)

Feature columns:
['age', 'bmi', 'blood_pressure', 'glucose', 'cholesterol', 'smoking']

Target distribution:


,Count
label,
0,2500
1,2500



TRAIN/TEST SPLIT
Training features: (4000, 6)
Testing features: (1000, 6)
Training labels: (4000,)
Testing labels: (1000,)

Training label percentages:


,Percentage
label,
0,50.0
1,50.0



Testing label percentages:


,Percentage
label,
0,50.0
1,50.0



Saved files:
c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\X_train.csv
c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\X_test.csv
c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\y_train.csv
c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\y_test.csv
c:\Users\saura\OneDrive\Desktop\early_disease_prediction\models\imputer.joblib
c:\Users\saura\OneDrive\Desktop\early_disease_prediction\models\scaler.joblib
c:\Users\saura\OneDrive\Desktop\early_disease_prediction\models\feature_columns.joblib

Data preprocessing completed.
